# CoT in Small Open LLMs: Pipeline (GSM8K, English vs Hindi)

**Har member ke liye:**
1. Runtime > Change runtime type > **GPU** on karo.
2. Shared folder `DL_CoT_Project` ko apne **My Drive** mein shortcut karo (Drive mein folder pe right click > Organize > Add shortcut > My Drive).
3. Sirf **Cell 1** mein `MODEL`, `MODEL_TAG`, `BS`, `USE_4BIT` badlo. Baaki settings team-wide hain, **badalni nahi**.
4. Cells upar se neeche chalao. Cell 7 chhota test hai (kuch save nahi hota), Cell 8 poore runs karta hai.

| Member | MODEL | MODEL_TAG |
|---|---|---|
| Priya | Qwen/Qwen2.5-0.5B-Instruct | qwen0.5b |
| Member 1 | Qwen/Qwen2.5-1.5B-Instruct | qwen1.5b |
| Member 2 | Qwen/Qwen2.5-3B-Instruct | qwen3b |
| Member 3 | Qwen/Qwen2.5-7B-Instruct (USE_4BIT = True) | qwen7b |

In [ ]:
# ===== CELL 1: SIRF YE HISSA BADLO (apna model) =====
MODEL     = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_TAG = "qwen0.5b"     # qwen0.5b / qwen1.5b / qwen3b / qwen7b
BS        = 16             # batch size. 3B: 8-16, 7B: 4-8 (memory error aaye toh kam karo)
USE_4BIT  = False          # 7B ke liye True

# ===== Team-wide settings: BADLO MAT =====
N              = 200       # questions
SEED           = 42
MAX_NEW_TOKENS = 512
SC_SAMPLES     = 5         # self-consistency samples
SC_TEMP        = 0.7       # self-consistency temperature
ROOT = "/content/drive/MyDrive/DL_CoT_Project/"

In [ ]:
# ===== CELL 2: Drive mount + paths =====
from google.colab import drive
drive.mount("/content/drive")

import os, re, json, time, glob
from collections import Counter

DATA = ROOT + "data/"
RES  = ROOT + "results/"
assert os.path.isdir(ROOT), "DL_CoT_Project folder nahi mila. Shared folder ka shortcut My Drive mein add karo."
os.makedirs(RES, exist_ok=True)
print("data/ mein:", os.listdir(DATA))

In [ ]:
# ===== CELL 3: Model load =====
!pip install -q transformers accelerate bitsandbytes
import torch
from transformers import pipeline, BitsAndBytesConfig

if USE_4BIT:
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    pipe = pipeline("text-generation", model=MODEL, device_map="auto",
                    model_kwargs={"quantization_config": q})
else:
    try:
        pipe = pipeline("text-generation", model=MODEL, device_map="auto", dtype="auto")
    except Exception:
        pipe = pipeline("text-generation", model=MODEL, device_map="auto", torch_dtype="auto")

pipe.tokenizer.padding_side = "left"
if pipe.tokenizer.pad_token is None:
    pipe.tokenizer.pad_token = pipe.tokenizer.eos_token
print("Model ready:", MODEL)

In [ ]:
# ===== CELL 4: Answer extraction =====
HI_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
NUM = r"-?\d+(?:\.\d+)?"

def extract(text):
    """Model ke output se final number nikalta hai (string). Na mile toh None."""
    text = text.translate(HI_DIGITS).replace(",", "")
    m = re.findall(r"answer is[\s:*$\\(]*(" + NUM + ")", text, flags=re.I)
    if m:
        return m[-1]
    m = re.findall(r"\\boxed\{\s*\$?(" + NUM + ")", text)
    if m:
        return m[-1]
    nums = re.findall(NUM, text)
    return nums[-1] if nums else None

def norm(p):
    """'26', '26.0', '26.' sab ko ek jaisa banata hai (majority vote ke liye zaroori)."""
    f = float(p)
    return str(int(f)) if f == int(f) else str(f)

def same(pred, gt):
    try:
        return abs(float(pred) - float(gt)) < 1e-6
    except (TypeError, ValueError):
        return False

In [ ]:
# ===== CELL 5: Prompts (4 strategies, English + Hindi) =====
# Rule: instruction hamesha English mein, question language ke hisaab se (en ya hi).
def clean_final(ans):
    """Few-shot example ke answer se sirf final number (GSM8K '#### 72' format)."""
    if "####" in ans:
        return ans.split("####")[-1].strip().replace(",", "")
    return extract(ans)

def build_prompt(q, strategy, fewshot=None):
    if strategy == "zeroshot":
        return q + "\nDo not show any steps. Give only the final answer in this form: The answer is <number>."
    if strategy in ("cot", "selfcons"):
        return q + "\nLet's think step by step. End your response with: The answer is <number>."
    if strategy == "fewshot":
        shots = "".join(f"Question: {e['question']}\nThe answer is {clean_final(e['answer'])}.\n\n"
                        for e in fewshot)
        return shots + f"Question: {q}\nGive only the final answer in this form: The answer is <number>."
    raise ValueError("unknown strategy: " + strategy)

In [ ]:
# ===== CELL 6: Experiment engine =====
def load_data(lang):
    data = json.load(open(DATA + f"gsm8k_200_{lang}.json", encoding="utf-8"))
    assert len(data) == 200, f"{lang}: 200 entries honi chahiye, mili {len(data)}"
    return data

def load_fewshot(lang):
    fs = json.load(open(DATA + f"fewshot_{lang}.json", encoding="utf-8"))
    assert len(fs) == 4, f"fewshot_{lang}: 4 examples honi chahiye, mile {len(fs)}"
    return fs

def generate_batch(prompts, sample=False):
    chats = [[{"role": "user", "content": p}] for p in prompts]
    gen = dict(do_sample=True, temperature=SC_TEMP) if sample else dict(do_sample=False)
    res = pipe(chats, batch_size=BS, max_new_tokens=MAX_NEW_TOKENS, **gen)
    outs = []
    for r in res:
        r = r[0] if isinstance(r, list) else r
        outs.append(r["generated_text"][-1]["content"])
    return outs

def run_experiment(strategy, lang, n=N, save=True):
    if save and n != N:
        print(f"WARNING: n={n} (N={N} nahi), isliye file save nahi hogi.")
        save = False
    data = load_data(lang)[:n]
    fs = load_fewshot(lang) if strategy == "fewshot" else None
    prompts = [build_prompt(ex["question"], strategy, fs) for ex in data]
    torch.manual_seed(SEED)
    t0 = time.time()

    items, correct = [], 0
    if strategy == "selfcons":
        samples = [generate_batch(prompts, sample=True) for _ in range(SC_SAMPLES)]
        for ex, outs in zip(data, zip(*samples)):
            preds = [extract(o) for o in outs]
            votes = [norm(p) for p in preds if p is not None]
            pred = Counter(votes).most_common(1)[0][0] if votes else None
            ok = same(pred, ex["answer"])
            correct += ok
            items.append({"id": ex["id"], "outputs": list(outs), "preds": preds,
                          "pred": pred, "gt": ex["answer"], "correct": bool(ok)})
    else:
        outs = generate_batch(prompts)
        for ex, o in zip(data, outs):
            pred = extract(o)
            ok = same(pred, ex["answer"])
            correct += ok
            items.append({"id": ex["id"], "output": o, "pred": pred, "gt": ex["answer"],
                          "correct": bool(ok), "format_ok": "answer is" in o.lower()})

    acc = correct / len(data)
    result = {"model": MODEL, "model_tag": MODEL_TAG, "strategy": strategy, "lang": lang,
              "n": len(data), "accuracy": acc, "correct": int(correct),
              "settings": {"seed": SEED, "max_new_tokens": MAX_NEW_TOKENS, "batch_size": BS,
                           "sc_samples": SC_SAMPLES, "sc_temp": SC_TEMP, "4bit": USE_4BIT},
              "seconds": round(time.time() - t0), "items": items}
    name = f"{MODEL_TAG}_{strategy}_{lang}.json"
    if save:
        json.dump(result, open(RES + name, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"{name}: {correct}/{len(data)} = {100*acc:.1f}%  ({result['seconds']} sec)"
          + ("" if save else "  [test, save nahi hua]"))
    return result

In [ ]:
# ===== CELL 7: Chhota test (10 questions, kuch save nahi hota) =====
# Pehle yahi chalao. Score jo bhi aaye, bas errors nahi aani chahiye.
for strat in ["zeroshot", "cot", "fewshot"]:
    run_experiment(strat, "en", n=10, save=False)

# Ek output khud padh ke dekho (extraction sahi hai ya nahi):
r = run_experiment("cot", "en", n=3, save=False)
for it in r["items"]:
    print("-" * 60)
    print(it["output"][-300:])
    print("EXTRACTED:", it["pred"], "| GT:", it["gt"])

In [ ]:
# ===== CELL 8: POORE RUNS (200 questions, English + Hindi) =====
# Jo file ban chuki hai use skip karta hai, toh session beech mein band ho jaye
# toh Cell 1-6 dobara chalake yeh cell chalao, wahin se aage badhega.
# Hindi runs tabhi chalenge jab data/ mein gsm8k_200_hi.json aur fewshot_hi.json ho.
ORDER = ["zeroshot", "cot", "fewshot", "selfcons"]     # jaldi wale pehle, selfcons sabse last

for lang in ["en", "hi"]:
    if not os.path.exists(DATA + f"gsm8k_200_{lang}.json"):
        print(f"[{lang}] data file abhi ready nahi, skip.")
        continue
    for strat in ORDER:
        if os.path.exists(RES + f"{MODEL_TAG}_{strat}_{lang}.json"):
            print(f"skip (already done): {MODEL_TAG}_{strat}_{lang}.json")
            continue
        run_experiment(strat, lang)

In [ ]:
# ===== CELL 9: Apne model ke results ka summary =====
import pandas as pd
rows = []
for f in sorted(glob.glob(RES + f"{MODEL_TAG}_*.json")):
    r = json.load(open(f, encoding="utf-8"))
    rows.append({"strategy": r["strategy"], "lang": r["lang"], "acc%": round(100 * r["accuracy"], 1)})
if rows:
    print(pd.DataFrame(rows).pivot(index="strategy", columns="lang", values="acc%"))
print(len(rows), "/ 8 files ready")